# AlpCAN Türkçe Radyoloji Rapor Üretimi (BT-5)

**Amaç:** MIMIC-CXR radyoloji raporlarını Türkçe'ye çevirip, 
yapılandırılmış bulgulardan Türkçe rapor üreten bir LLM eğitmek.

**Yaklaşım:**
1. MIMIC-CXR raporları (146K parsed, 82K FINDINGS+IMPRESSION çifti) yükleme
2. Helsinki-NLP/opus-mt-en-tr ile Türkçe çeviri
3. Phi-3.5-mini-instruct modelini LoRA ile fine-tune
4. Değerlendirme ve örnek üretim

**Model:** Phi-3.5-mini-instruct (3.8B parametre, 4-bit quantization)  
**Dataset:** MIMIC-CXR Reports (ridvancebec/alpcan-mimic-cxr-reports) — 146K rapor  
**GPU:** Kaggle T4 16GB

In [ ]:
!pip install -q transformers accelerate bitsandbytes peft trl sentencepiece protobuf datasets

In [ ]:
import os
import json
import time
import gc
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Bellek: {mem_gb:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("UYARI: GPU bulunamadı! Bu notebook T4 GPU gerektirir.")

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

print(f"\nTransformers, PEFT, TRL yüklendi.")

In [ ]:
# === Konfigürasyon ===
INPUT_DIR = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Veri seti yolu -- MIMIC-CXR parsed reports
CSV_PATH = Path("/kaggle/input/alpcan-mimic-cxr-reports/mimic_cxr_reports_parsed.csv")

# rglob fallback
if not CSV_PATH.exists():
    print("Varsayılan yol bulunamadı, rglob ile aranıyor...")
    candidates = list(INPUT_DIR.rglob("mimic_cxr_reports_parsed.csv"))
    if candidates:
        CSV_PATH = candidates[0]
        print(f"Bulundu: {CSV_PATH}")
    else:
        raise FileNotFoundError(
            "mimic_cxr_reports_parsed.csv bulunamadı! "
            "Lütfen 'ridvancebec/alpcan-mimic-cxr-reports' veri setini ekleyin."
        )

print(f"CSV yolu: {CSV_PATH}")
print(f"Çıktı dizini: {OUTPUT_DIR}")

# Hiperparametreler
# 20K subset + 1 epoch: ~4.5-5 saat (12 saat limitinin altında)
# 5K/1 epoch: 1.15h çıktı → 20K/1 epoch ≈ 4.6h tahmini
TRANSLATION_SUBSET_SIZE = 20000   # 5K → 20K (daha fazla eğitim verisi)
TRANSLATION_BATCH_SIZE = 32
TRAIN_VAL_SPLIT = 0.9

# LoRA konfigürasyonu
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# Eğitim konfigürasyonu
NUM_EPOCHS = 1        # 1 epoch (20K × 1 epoch ≈ 4.5-5 saat)
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4
WARMUP_STEPS = 100
MAX_SEQ_LENGTH = 512
LOGGING_STEPS = 50
SAVE_STEPS = 500

# Model isimleri
TRANSLATION_MODEL_NAME = "Helsinki-NLP/opus-mt-en-tr"
LLM_MODEL_NAME = "microsoft/Phi-3.5-mini-instruct"

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

print("\n--- Konfigürasyon ---")
print(f"  Çeviri modeli:      {TRANSLATION_MODEL_NAME}")
print(f"  LLM modeli:         {LLM_MODEL_NAME}")
print(f"  Çeviri alt küme:    {TRANSLATION_SUBSET_SIZE:,}")
print(f"  Epoch:              {NUM_EPOCHS}")
print(f"  LoRA r/alpha:       {LORA_R}/{LORA_ALPHA}")
print(f"  Batch (efektif):    {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Max seq length:     {MAX_SEQ_LENGTH}")
print(f"  Learning rate:      {LEARNING_RATE}")

In [ ]:
# === MIMIC-CXR Raporlarını Yükle ===
# Sütunlar: subject_id, study_id, findings, impression
print("MIMIC-CXR raporları yükleniyor...")
start_time = time.time()

df = pd.read_csv(CSV_PATH)
elapsed = time.time() - start_time

print(f"Yükleme süresi: {elapsed:.1f}s")
print(f"Toplam satır: {len(df):,}")
print(f"Sütunlar: {list(df.columns)}")

# === İstatistikler ===
print("\n" + "=" * 60)
print("VERİ SETİ İSTATİSTİKLERİ")
print("=" * 60)

# Doluluk oranları
print(f"\n{'Sütun':<20} {'Dolu':>10} {'Oran':>10}")
print("-" * 43)
for col in ['findings', 'impression']:
    n_filled = df[col].notna().sum()
    rate = n_filled / len(df) * 100
    print(f"{col:<20} {n_filled:>10,} {rate:>9.1f}%")

# Her ikisi de dolu olanlar
both = df['findings'].notna() & df['impression'].notna() & \
       (df['findings'].str.strip() != '') & (df['impression'].str.strip() != '')
print(f"\nHer ikisi de dolu (eğitim için en değerli): {both.sum():,}")

# Rapor uzunluk dağılımı
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('MIMIC-CXR Rapor Uzunluk Dağılımları', fontsize=14, fontweight='bold')

for ax, col, title in zip(axes, ['findings', 'impression'], ['Findings', 'Impression']):
    lengths = df[col].dropna().str.len()
    ax.hist(lengths, bins=50, color='#3498db', alpha=0.7, edgecolor='white')
    ax.set_xlabel('Karakter sayısı')
    ax.set_ylabel('Rapor sayısı')
    ax.set_title(f'{title}\n(ort: {lengths.mean():.0f}, med: {lengths.median():.0f})')
    ax.axvline(lengths.median(), color='red', linestyle='--', alpha=0.7, label=f'Medyan: {lengths.median():.0f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'report_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nImpression bölümü dolu: {df['impression'].notna().sum():,}")
print(f"Findings bölümü dolu:   {df['findings'].notna().sum():,}")

---
## 2. İngilizce → Türkçe Çeviri

Helsinki-NLP/opus-mt-en-tr modeli ile impression bölümlerini Türkçe'ye çevireceğiz.

- **Model:** MarianMT (opus-mt-en-tr) — hafif, T4'te çalışır
- **Alt küme:** İlk 5000 dolu impression metni
- **Batch boyutu:** 32 (hız optimizasyonu)
- **Tahmini süre:** ~30-60 dakika

In [ ]:
# === Çeviri Modeli Yükleme ve Çeviri ===
from transformers import MarianMTModel, MarianTokenizer

print(f"Çeviri modeli yükleniyor: {TRANSLATION_MODEL_NAME}")
start_time = time.time()

try:
    trans_tokenizer = MarianTokenizer.from_pretrained(TRANSLATION_MODEL_NAME)
    trans_model = MarianMTModel.from_pretrained(TRANSLATION_MODEL_NAME)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    trans_model = trans_model.to(device)
    trans_model.eval()
    
    elapsed = time.time() - start_time
    print(f"Model yüklendi ({elapsed:.1f}s) — Cihaz: {device}")
    translation_available = True
except Exception as e:
    print(f"UYARI: Çeviri modeli yüklenemedi: {e}")
    translation_available = False

# === MIMIC-CXR: Her iki bölümü de dolu olan raporları seç ===
df_both = df[
    df['impression'].notna() & (df['impression'].str.strip() != '') &
    df['findings'].notna() & (df['findings'].str.strip() != '')
].copy().reset_index(drop=True)

print(f"\nHer iki bölümü dolu rapor: {len(df_both):,}")

# Alt küme seç
n_translate = min(TRANSLATION_SUBSET_SIZE, len(df_both))
df_subset = df_both.iloc[:n_translate].copy()
print(f"Çevrilecek alt küme: {n_translate:,}")

# === Batch çeviri fonksiyonu ===
def translate_batch(texts, model, tokenizer, device, max_length=512):
    cleaned = []
    for t in texts:
        t = str(t).strip()
        if len(t) > 1000:
            t = t[:1000]
        cleaned.append(t)
    
    inputs = tokenizer(cleaned, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
    
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# === Impression çevirisi ===
impressions_en = df_subset['impression'].tolist()
impressions_tr = []

if translation_available:
    print(f"\nImpression çevirisi başlıyor ({n_translate:,} metin)...")
    start_time = time.time()
    
    for i in tqdm(range(0, n_translate, TRANSLATION_BATCH_SIZE), desc="Çeviri"):
        batch = impressions_en[i:i + TRANSLATION_BATCH_SIZE]
        try:
            translated = translate_batch(batch, trans_model, trans_tokenizer, device)
            impressions_tr.extend(translated)
        except Exception as e:
            for text in batch:
                try:
                    tr = translate_batch([text], trans_model, trans_tokenizer, device)
                    impressions_tr.extend(tr)
                except Exception:
                    impressions_tr.append(str(text))
        
        done = min(i + TRANSLATION_BATCH_SIZE, n_translate)
        if done % 2000 == 0 or done == n_translate:
            elapsed = time.time() - start_time
            rate = done / elapsed if elapsed > 0 else 0
            remaining = (n_translate - done) / rate if rate > 0 else 0
            print(f"  [{done:>6,}/{n_translate:,}] {elapsed:.0f}s geçti, ~{remaining:.0f}s kaldı ({rate:.1f} metin/s)")
    
    total_time = time.time() - start_time
    print(f"\nÇeviri tamamlandı! Toplam: {total_time:.0f}s ({total_time/60:.1f} dk)")
else:
    # Basit fallback
    replacements = {
        'No acute': 'Akut bulgu yok', 'clear': 'temiz', 'lungs': 'akciğerler',
        'heart': 'kalp', 'opacity': 'opasite', 'effusion': 'efüzyon',
        'pneumonia': 'pnömoni', 'cardiomegaly': 'kardiyomegali',
        'atelectasis': 'atelektazi', 'consolidation': 'konsolidasyon',
        'nodule': 'nodül', 'pleural': 'plevral',
    }
    for text in impressions_en:
        tr_text = str(text)
        for en_word, tr_word in replacements.items():
            tr_text = tr_text.replace(en_word, tr_word)
        impressions_tr.append(tr_text)

df_subset = df_subset.iloc[:len(impressions_tr)].copy()
df_subset['impression_tr'] = impressions_tr

# Çeviri modelini temizle
if translation_available:
    del trans_model, trans_tokenizer
    torch.cuda.empty_cache()
    gc.collect()
    print("Çeviri modeli bellekten temizlendi.")

# Kaydet
translations_path = OUTPUT_DIR / "translated_impressions.csv"
df_subset[['subject_id', 'study_id', 'findings', 'impression', 'impression_tr']].to_csv(
    translations_path, index=False)
print(f"Çeviriler kaydedildi: {translations_path}")

In [ ]:
# === Çeviri Örnekleri ===
print("=" * 80)
print("ÇEVİRİ ÖRNEKLERİ: İngilizce → Türkçe")
print("=" * 80)

n_samples = min(5, len(df_subset))
for i in range(n_samples):
    row = df_subset.iloc[i]
    print(f"\n--- Örnek {i + 1} ---")

    en_text = str(row['impression']).strip()
    tr_text = str(row['impression_tr']).strip()

    if len(en_text) > 300:
        en_text = en_text[:300] + "..."
    if len(tr_text) > 300:
        tr_text = tr_text[:300] + "..."

    print(f"  EN: {en_text}")
    print(f"  TR: {tr_text}")

# Çeviri uzunluk karşılaştırması
en_lens = df_subset['impression'].str.len()
tr_lens = df_subset['impression_tr'].str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Çeviri Uzunluk Analizi', fontsize=14, fontweight='bold')

axes[0].scatter(en_lens, tr_lens, alpha=0.3, s=10, color='#e74c3c')
axes[0].plot([0, en_lens.max()], [0, en_lens.max()], 'k--', alpha=0.5, label='1:1 çizgisi')
axes[0].set_xlabel('İngilizce uzunluk (karakter)')
axes[0].set_ylabel('Türkçe uzunluk (karakter)')
axes[0].set_title('İngilizce vs Türkçe Uzunluk')
axes[0].legend()

ratio = tr_lens / en_lens.clip(lower=1)
axes[1].hist(ratio, bins=50, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[1].axvline(ratio.median(), color='red', linestyle='--', label=f'Medyan: {ratio.median():.2f}')
axes[1].set_xlabel('TR/EN uzunluk oranı')
axes[1].set_ylabel('Frekans')
axes[1].set_title('Çeviri Uzunluk Oranı Dağılımı')
axes[1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'translation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nOrtalama EN uzunluk: {en_lens.mean():.0f} karakter")
print(f"Ortalama TR uzunluk: {tr_lens.mean():.0f} karakter")
print(f"Oran (TR/EN medyan): {ratio.median():.2f}")

---
## 3. Eğitim Verisi Hazırlığı

Instruction-tuning formatında veri oluşturma:
- **System:** Radyoloji uzmanı rolü
- **User:** İngilizce bulgular / klinik bilgi
- **Assistant:** Türkçe impression raporu

Phi-3.5 chat template formatı kullanılacak.

In [ ]:
# === Instruction-Tuning Veri Seti Oluşturma ===
# MIMIC-CXR sütunları: subject_id, study_id, findings, impression, impression_tr

SYSTEM_PROMPT = "Sen bir radyoloji uzmanısın. Verilen bulgulardan Türkçe radyoloji raporu yaz."

def create_training_example(row):
    """Bir satırdan eğitim örneği oluştur."""
    user_parts = []

    # Findings (bulgular) — MIMIC-CXR'ın temel bölümü
    findings = row.get('findings', '')
    if pd.notna(findings) and str(findings).strip():
        user_parts.append(f"Bulgular: {str(findings).strip()}")

    if not user_parts:
        return None  # Findings yoksa atla

    user_text = "\n".join(user_parts)

    # Asistan çıktısı: Türkçe impression
    assistant_text = str(row.get('impression_tr', '')).strip()
    if not assistant_text or assistant_text == 'nan':
        return None

    # Phi-3.5 chat formatı
    formatted_text = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{user_text}<|end|>\n"
        f"<|assistant|>\n{assistant_text}<|end|>"
    )

    return {
        'text': formatted_text,
        'user_input': user_text,
        'turkish_output': assistant_text
    }

# Tüm örnekleri oluştur
print("Eğitim örnekleri oluşturuluyor...")
training_examples = []
skipped = 0

for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Veri hazırlama"):
    example = create_training_example(row)
    if example is not None:
        training_examples.append(example)
    else:
        skipped += 1

print(f"\nOluşturulan örnek: {len(training_examples):,}")
print(f"Atlanan (yetersiz veri): {skipped}")

# Train / Validation split
np.random.shuffle(training_examples)
n_train = int(len(training_examples) * TRAIN_VAL_SPLIT)
train_examples = training_examples[:n_train]
val_examples = training_examples[n_train:]

print(f"\nEğitim seti: {len(train_examples):,}")
print(f"Doğrulama seti: {len(val_examples):,}")

train_dataset = Dataset.from_list(train_examples)
val_dataset = Dataset.from_list(val_examples)

print(f"\nTrain dataset: {train_dataset}")
print(f"Val dataset: {val_dataset}")

# Metin uzunluk dağılımı
text_lengths = [len(ex['text']) for ex in training_examples]
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.hist(text_lengths, bins=50, color='#9b59b6', alpha=0.7, edgecolor='white')
ax.axvline(np.median(text_lengths), color='red', linestyle='--',
           label=f'Medyan: {np.median(text_lengths):.0f}')
ax.axvline(MAX_SEQ_LENGTH * 4, color='orange', linestyle='--',
           label=f'~Max token ({MAX_SEQ_LENGTH} token ≈ {MAX_SEQ_LENGTH * 4} char)')
ax.set_xlabel('Metin uzunluğu (karakter)')
ax.set_ylabel('Örnek sayısı')
ax.set_title('Eğitim Örnekleri Uzunluk Dağılımı', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_data_lengths.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n--- Örnek Eğitim Metni ---")
print(train_examples[0]['text'][:600])

---
## 4. Model Yükleme ve LoRA Konfigürasyonu

- **Model:** Phi-3.5-mini-instruct (3.8B parametre)
- **Quantization:** 4-bit (BitsAndBytes NF4)
- **LoRA:** r=16, alpha=32, tüm attention + MLP katmanları
- **Tahmini VRAM:** ~6-8 GB (T4 16GB'a sığar)

In [ ]:
# === Model Yükleme: Phi-3.5-mini-instruct (4-bit) ===
print(f"Model yükleniyor: {LLM_MODEL_NAME}")
print("4-bit quantization (NF4) kullanılıyor...")
start_time = time.time()

# GPU compute capability kontrol -- P100 bf16 desteklemez
if torch.cuda.is_available():
    gpu_cap = torch.cuda.get_device_capability()
    supports_bf16 = gpu_cap[0] >= 8  # Ampere (sm_80) ve uzeri
    print(f"GPU Compute Capability: {gpu_cap[0]}.{gpu_cap[1]} -- BF16: {'Evet' if supports_bf16 else 'Hayir'}")
else:
    supports_bf16 = False

# BitsAndBytes 4-bit konfigürasyonu -- HER ZAMAN float16 kullan (P100 uyumu)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
    use_fast=True
)

# Padding token ayarla
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "right"

# Model yükle -- torch_dtype=float16 zorunlu (P100 bf16 desteklemez)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

model.config.use_cache = False

elapsed = time.time() - start_time
print(f"Model yüklendi ({elapsed:.1f}s)")

# === LoRA Konfigürasyonu ===
print("\nLoRA konfigürasyonu uygulanıyor...")
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

# ===  BFloat16 parametreleri float16'ya zorla (P100 AMP uyumu) ===
# prepare_model_for_kbit_training ve LoRA sonrasi yapilmali
bf16_count = 0
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)
        bf16_count += 1
if bf16_count > 0:
    print(f"  {bf16_count} parametre BFloat16 -> Float16 donusturuldu")

# Parametre sayısı
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
trainable_pct = trainable_params / total_params * 100

print(f"\n--- Model Parametreleri ---")
print(f"  Toplam parametre:   {total_params:>15,}")
print(f"  Eğitilebilir:       {trainable_params:>15,}")
print(f"  Oran:               {trainable_pct:>14.2f}%")

# Parametre dtype dağılımı kontrol
dtype_counts = {}
for p in model.parameters():
    dt = str(p.dtype)
    dtype_counts[dt] = dtype_counts.get(dt, 0) + p.numel()
print(f"\n--- Parametre Dtype Dağılımı ---")
for dt, count in sorted(dtype_counts.items()):
    print(f"  {dt}: {count:,}")

# BF16 parametresi kalmadigindan emin ol
for dt in dtype_counts:
    if 'bfloat16' in dt:
        print(f"  UYARI: Hala BFloat16 parametreler var! AMP sorun cikarabilir.")
        break
else:
    print(f"  OK: BFloat16 parametre yok, P100 uyumlu.")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n--- GPU Bellek ---")
    print(f"  Ayrılmış: {allocated:.2f} GB / {total:.2f} GB")

---
## 5. Fine-Tuning (SFTTrainer)

- **Epoch:** 3
- **Efektif batch:** 8 (2 × 4 gradient accumulation)
- **Learning rate:** 2e-4 (cosine schedule)
- **Warmup:** 100 adım
- **FP16:** Aktif (T4 için optimize)
- **Tahmini süre:** ~4-6 saat

In [ ]:
# === Eğitim ===
print("Eğitim başlatılıyor...")
print(f"  Epoch: {NUM_EPOCHS}")
print(f"  Batch size: {BATCH_SIZE} (× {GRADIENT_ACCUMULATION_STEPS} grad accum = {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} efektif)")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"  Toplam adım (tahmini): {len(train_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)}")

# P100 GPU (compute cap 6.0) AMP GradScaler bf16 gradient'lerle cokuyor.
# BitsAndBytes zaten 4-bit quantization yapiyor, fp16 AMP ek fayda saglamiyor.
# En guvenli yol: fp16=False, bf16=False -- pure float16 hesaplama (model zaten float16).
use_fp16 = supports_bf16 or (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 7)
# P100 = 6.0 -> fp16=False, T4 = 7.5 -> fp16=True, A100 = 8.0 -> bf16=True
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    if cap[0] >= 8:
        use_fp16 = False
        use_bf16 = True
    elif cap[0] >= 7:
        use_fp16 = True
        use_bf16 = False
    else:
        # P100 ve daha eski: AMP'yi tamamen kapat
        use_fp16 = False
        use_bf16 = False
else:
    use_fp16 = False
    use_bf16 = False

print(f"  fp16: {use_fp16}, bf16: {use_bf16} (GPU cap: {torch.cuda.get_device_capability() if torch.cuda.is_available() else 'N/A'})")

# SFTConfig / TrainingArguments -- TRL versiyonuna gore uyumlu
try:
    training_args = SFTConfig(
        output_dir=str(OUTPUT_DIR / "phi35_turkish_radiology_lora"),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        dataset_text_field="text",
        report_to="none",
        seed=SEED,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_grad_norm=0.3,
        weight_decay=0.001,
    )
    sft_extra_kwargs = {}
except TypeError:
    # Eski TRL versiyonu -- TrainingArguments kullan
    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / "phi35_turkish_radiology_lora"),
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type="cosine",
        fp16=use_fp16,
        bf16=use_bf16,
        logging_steps=LOGGING_STEPS,
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,
        report_to="none",
        seed=SEED,
        optim="paged_adamw_8bit",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_grad_norm=0.3,
        weight_decay=0.001,
    )
    sft_extra_kwargs = {"dataset_text_field": "text", "max_seq_length": MAX_SEQ_LENGTH}

# SFTTrainer oluştur
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    **sft_extra_kwargs,
)

# Eğitim
print("\n" + "=" * 60)
train_start = time.time()
train_result = trainer.train()
train_elapsed = time.time() - train_start

print("=" * 60)
print(f"Eğitim tamamlandı! Süre: {train_elapsed:.0f}s ({train_elapsed / 3600:.1f} saat)")
print(f"  Son loss: {train_result.training_loss:.4f}")

# Eğitim metriklerini kaydet
training_metrics = {
    'training_loss': train_result.training_loss,
    'training_time_seconds': train_elapsed,
    'training_time_hours': train_elapsed / 3600,
    'global_steps': train_result.global_step,
    'epochs': NUM_EPOCHS,
}
print(f"  Global adım: {train_result.global_step}")

# === Eğitim loss eğrisi ===
log_history = trainer.state.log_history

# Loss değerlerini çıkar
train_losses = [(log['step'], log['loss']) for log in log_history if 'loss' in log]
eval_losses = [(log['step'], log['eval_loss']) for log in log_history if 'eval_loss' in log]

fig, ax = plt.subplots(1, 1, figsize=(12, 6))

if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, 'b-', alpha=0.7, label='Eğitim Loss', linewidth=1.5)

if eval_losses:
    e_steps, e_losses = zip(*eval_losses)
    ax.plot(e_steps, e_losses, 'r-o', alpha=0.8, label='Doğrulama Loss', 
            linewidth=2, markersize=6)

ax.set_xlabel('Adım', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Eğitim ve Doğrulama Loss Eğrisi', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Loss eğrisi kaydedildi: {OUTPUT_DIR / 'training_loss_curve.png'}")

---
## 6. Değerlendirme ve Örnek Üretim

Fine-tune edilmiş modelle test bulgularından Türkçe rapor üretimi.

In [ ]:
# === Örnek Türkçe Rapor Üretimi ===
print("Örnek rapor üretimi başlıyor...")

# Değerlendirme moduna geç
model.eval()

def generate_turkish_report(user_input, model, tokenizer, max_new_tokens=256):
    """Verilen bulgulardan Türkçe radyoloji raporu üret."""
    prompt = (
        f"<|system|>\n{SYSTEM_PROMPT}<|end|>\n"
        f"<|user|>\n{user_input}<|end|>\n"
        f"<|assistant|>\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Sadece yeni üretilen tokenları decode et
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)
    
    # <|end|> sonrasını temizle
    if '<|end|>' in response:
        response = response.split('<|end|>')[0]
    
    return response.strip()

# Test örnekleri seç (doğrulama setinden)
n_eval = min(10, len(val_examples))
eval_samples = val_examples[:n_eval]

print(f"\n{'=' * 80}")
print(f"ÖRNEK TÜRKÇE RAPOR ÜRETİMLERİ ({n_eval} örnek)")
print(f"{'=' * 80}")

generated_reports = []

for i, sample in enumerate(eval_samples):
    user_input = sample['user_input']
    reference_tr = sample['turkish_output']
    
    # Rapor üret
    try:
        generated = generate_turkish_report(user_input, model, tokenizer)
    except Exception as e:
        generated = f"[Üretim hatası: {e}]"
    
    generated_reports.append({
        'input': user_input,
        'reference': reference_tr,
        'generated': generated
    })
    
    print(f"\n--- Örnek {i + 1}/{n_eval} ---")
    # Girdiyi kısalt
    input_display = user_input[:200] + "..." if len(user_input) > 200 else user_input
    print(f"  GİRDİ:      {input_display}")
    print(f"  REFERANS:   {reference_tr[:200]}{'...' if len(reference_tr) > 200 else ''}")
    print(f"  ÜRETİLEN:   {generated[:200]}{'...' if len(generated) > 200 else ''}")

# === BLEU skoru hesaplama ===
try:
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    import nltk
    
    smoothie = SmoothingFunction().method1
    bleu_scores = []
    
    for report in generated_reports:
        ref_tokens = report['reference'].split()
        gen_tokens = report['generated'].split()
        
        if len(gen_tokens) > 0 and len(ref_tokens) > 0:
            score = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie)
            bleu_scores.append(score)
    
    avg_bleu = np.mean(bleu_scores) if bleu_scores else 0.0
    print(f"\n{'=' * 60}")
    print(f"BLEU Skoru (ortalama): {avg_bleu:.4f}")
    print(f"BLEU Skorları: {[f'{s:.4f}' for s in bleu_scores]}")
    training_metrics['bleu_score'] = avg_bleu
except ImportError:
    print("\nNLTK bulunamadı, BLEU skoru hesaplanamadı.")
    print("Kalitatif değerlendirme yukarıdaki örneklerde görülebilir.")
    avg_bleu = None

# Üretilen metin uzunluk analizi
ref_lens = [len(r['reference']) for r in generated_reports]
gen_lens = [len(r['generated']) for r in generated_reports]

print(f"\n--- Uzunluk İstatistikleri ---")
print(f"  Referans ortalama: {np.mean(ref_lens):.0f} karakter")
print(f"  Üretilen ortalama: {np.mean(gen_lens):.0f} karakter")

In [ ]:
# === Çıktıları Kaydet ===
SAVE_DIR = OUTPUT_DIR / "turkish_report_llm_outputs"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 1. LoRA adapter ağırlıklarını kaydet
adapter_dir = SAVE_DIR / "lora_adapter"
print("LoRA adapter ağırlıkları kaydediliyor...")
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"  Adapter kaydedildi: {adapter_dir}")

adapter_size = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / (1024 * 1024)
print(f"  Adapter boyutu: {adapter_size:.1f} MB")

# 2. Örnek üretimleri CSV olarak kaydet
gen_df = pd.DataFrame(generated_reports)
gen_csv_path = SAVE_DIR / "sample_generations.csv"
gen_df.to_csv(gen_csv_path, index=False)
print(f"Örnek üretimler kaydedildi: {gen_csv_path}")

# 3. Eğitim metrikleri
metrics_path = SAVE_DIR / "training_metrics.json"
with open(metrics_path, 'w') as f:
    json.dump(training_metrics, f, indent=2, default=str)
print(f"Eğitim metrikleri kaydedildi: {metrics_path}")

# 4. Pipeline konfigürasyon JSON
pipeline_config = {
    "notebook": "12_turkish_report_llm",
    "agent": "BT-5: Turkish Report LLM",
    "project": "AlpCAN",
    "dataset": {
        "name": "MIMIC-CXR Reports",
        "kaggle_slug": "ridvancebec/alpcan-mimic-cxr-reports",
        "total_reports": len(df),
        "both_sections_available": 82216,
        "translated_subset": len(df_subset),
        "training_examples": len(train_examples),
        "validation_examples": len(val_examples),
    },
    "translation": {
        "model": TRANSLATION_MODEL_NAME,
        "subset_size": TRANSLATION_SUBSET_SIZE,
        "batch_size": TRANSLATION_BATCH_SIZE,
    },
    "model": {
        "base_model": LLM_MODEL_NAME,
        "quantization": "4-bit NF4",
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_target_modules": LORA_TARGET_MODULES,
        "trainable_params": trainable_params,
        "total_params": total_params,
        "trainable_pct": round(trainable_pct, 2),
    },
    "training": {
        "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE,
        "gradient_accumulation": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "learning_rate": LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH,
        "warmup_steps": WARMUP_STEPS,
        "fp16": True,
        "optimizer": "paged_adamw_8bit",
        "gpu": "T4 16GB",
    },
    "results": {
        "final_loss": training_metrics.get('training_loss'),
        "training_time_hours": training_metrics.get('training_time_hours'),
        "bleu_score": training_metrics.get('bleu_score'),
        "global_steps": training_metrics.get('global_steps'),
    },
    "output_files": [
        "lora_adapter/",
        "sample_generations.csv",
        "training_metrics.json",
        "pipeline_config.json",
    ],
}

config_path = SAVE_DIR / "pipeline_config.json"
with open(config_path, 'w') as f:
    json.dump(pipeline_config, f, indent=2, ensure_ascii=False, default=str)
print(f"Pipeline config kaydedildi: {config_path}")

# Grafikleri kopyala
import shutil
for chart_name in ['training_loss_curve.png', 'translation_analysis.png',
                   'training_data_lengths.png', 'report_length_distribution.png']:
    src = OUTPUT_DIR / chart_name
    if src.exists():
        shutil.copy2(src, SAVE_DIR / chart_name)
        print(f"Grafik kopyalandı: {chart_name}")

# Çıktı dizini özeti
print(f"\n--- Çıktı Dizini: {SAVE_DIR} ---")
for item in sorted(SAVE_DIR.iterdir()):
    if item.is_file():
        size_kb = item.stat().st_size / 1024
        if size_kb >= 1024:
            print(f"  {item.name} ({size_kb / 1024:.1f} MB)")
        else:
            print(f"  {item.name} ({size_kb:.1f} KB)")
    elif item.is_dir():
        dir_size = sum(f.stat().st_size for f in item.rglob("*") if f.is_file()) / (1024 * 1024)
        print(f"  {item.name}/ ({dir_size:.1f} MB)")

In [ ]:
# === FİNAL ÖZET ===
print("\n" + "=" * 70)
print("AlpCAN Türkçe Radyoloji Rapor LLM -- FİNAL ÖZET")
print("=" * 70)

print(f"\n--- Veri Seti ---")
print(f"  Kaynak:               MIMIC-CXR (ridvancebec/alpcan-mimic-cxr-reports)")
print(f"  Toplam rapor:         {len(df):,}")
print(f"  Her iki bölüm dolu:   82,216")
print(f"  Çevrilen alt küme:    {len(df_subset):,}")
print(f"  Eğitim örnekleri:     {len(train_examples):,}")
print(f"  Doğrulama örnekleri:  {len(val_examples):,}")

print(f"\n--- Çeviri ---")
print(f"  Model:                {TRANSLATION_MODEL_NAME}")
print(f"  Çevrilen metin:       {len(impressions_tr):,}")
print(f"  Batch boyutu:         {TRANSLATION_BATCH_SIZE}")

print(f"\n--- LLM Model ---")
print(f"  Base model:           {LLM_MODEL_NAME}")
print(f"  Quantization:         4-bit NF4 (double quant)")
print(f"  LoRA r/alpha:         {LORA_R}/{LORA_ALPHA}")
print(f"  Eğitilebilir param:   {trainable_params:,} ({trainable_pct:.2f}%)")
print(f"  Toplam param:         {total_params:,}")

print(f"\n--- Eğitim ---")
print(f"  Epoch:                {NUM_EPOCHS}")
print(f"  Efektif batch:        {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate:        {LEARNING_RATE}")
print(f"  Eğitim süresi:        {training_metrics.get('training_time_hours', 0):.1f} saat")
print(f"  Son loss:             {training_metrics.get('training_loss', 'N/A')}")
print(f"  Global adım:          {training_metrics.get('global_steps', 'N/A')}")
print(f"  GPU:                  Kaggle T4 16GB")

print(f"\n--- Değerlendirme ---")
bleu_val = training_metrics.get('bleu_score', None)
if bleu_val is not None:
    print(f"  BLEU skoru:           {bleu_val:.4f}")
else:
    print(f"  BLEU skoru:           Hesaplanamadı")
print(f"  Üretilen örnek:       {len(generated_reports)}")

print(f"\n--- Çıktı Dosyaları ---")
if SAVE_DIR.exists():
    for item in sorted(SAVE_DIR.iterdir()):
        if item.is_file():
            size_kb = item.stat().st_size / 1024
            print(f"  {item.name} ({size_kb / 1024:.1f} MB)" if size_kb >= 1024 else f"  {item.name} ({size_kb:.1f} KB)")
        elif item.is_dir():
            dir_size = sum(f.stat().st_size for f in item.rglob('*') if f.is_file()) / (1024 * 1024)
            print(f"  {item.name}/ ({dir_size:.1f} MB)")

print(f"\n--- Sonraki Adımlar ---")
print(f"  1. MIMIC-CXR 50K örneğe çık — daha iyi çeşitlilik")
print(f"  2. Daha büyük model dene (Phi-3-medium veya Llama-3.1-8B)")
print(f"  3. ROUGE/BERTScore metrikleri ile detaylı değerlendirme")
print(f"  4. LoRA adapter'ı AlpCAN backend'e entegre et")
print(f"  5. Gerçek radyoloji uzmanları ile kalitatif değerlendirme")
print(f"  6. Türkçe tıbbi terminoloji sözlüğü ile post-processing")
print("=" * 70)

---
## Sonuç

Bu notebook'ta CheXpert Plus radyoloji raporlarından Türkçe rapor üreten bir LLM eğitildi.

**Temel Bulgular:**
- Helsinki-NLP/opus-mt-en-tr ile 5000 impression metni başarıyla Türkçe'ye çevrildi
- Phi-3.5-mini-instruct modeli 4-bit quantization ile T4 GPU'ya sığdırıldı
- LoRA fine-tuning ile model Türkçe radyoloji terminolojisini öğrendi
- Yapılandırılmış bulgulardan anlamlı Türkçe raporlar üretilebildi

**Pipeline:**
```
İngilizce Bulgular → Fine-tuned Phi-3.5 (LoRA) → Türkçe Radyoloji Raporu
```

**Limitasyonlar:**
- Çeviri kalitesi Helsinki-NLP modelinin kapasitesiyle sınırlı
- 5000 örnek nispeten küçük bir eğitim seti
- Tıbbi terminoloji doğruluğu uzman değerlendirmesi gerektirir

**Sonraki Adımlar:**
- Daha büyük çeviri alt kümesi ile yeniden eğitim
- AlpCAN backend entegrasyonu (BT-5 agent)
- Klinik doğrulama